# 01 · Dataset preview

A quick look at the sample customer-support dataset and at the embedding structure that makes the create-or-reuse trade-off non-trivial.

The stream mixes **well-covered** categories (good FAQ answers already exist → reuse is cheap) with **poorly-covered** categories (no FAQ entry → creating a reusable action pays off).

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
ROOT = ROOT if (ROOT / 'data').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('project root:', ROOT)

In [ ]:
import pandas as pd
faq = pd.read_csv(ROOT / 'data' / 'sample_faq_library.csv')
stream = pd.read_csv(ROOT / 'data' / 'sample_stream.csv')
print('FAQ actions:', len(faq), '| stream queries:', len(stream))
faq.head()

In [ ]:
stream.head(8)

In [ ]:
# how many incoming queries per category
stream['category'].value_counts()

## Embedding separation

Well-covered queries are close to an existing FAQ action; poorly-covered queries are far from everything in the library; and paraphrases inside a cluster are close to each other (so a created action is reusable for later, similar queries).

In [ ]:
import numpy as np
from genaction import Embedder
corpus = (list(faq['canonical_query']) + list(faq['response_text'])
          + list(stream['query_text']) + list(stream['gold_response']))
emb = Embedder(backend='tfidf').fit(corpus)
fe = emb.encode(list(faq['canonical_query']))
qe = emb.encode(list(stream['query_text']))
nn = np.clip(1 - (qe @ fe.T).max(axis=1), 0, 1)
s = stream.assign(nearest_faq_dist=nn)
well = ['billing','refund','account_login','troubleshooting','pricing','onboarding']
poor = ['subscription_cancel','sales_followup','crm_update','meeting_summary']
print('well-covered  mean nearest-FAQ distance: %.3f' % s[s.category.isin(well)].nearest_faq_dist.mean())
print('poorly-covered mean nearest-FAQ distance: %.3f' % s[s.category.isin(poor)].nearest_faq_dist.mean())

In [ ]:
# within-cluster paraphrase distance (what a created action exploits)
wc = []
for _, grp in s.groupby('gold_response'):
    idx = grp.index.tolist()
    if len(idx) < 2:
        continue
    g = qe[idx]
    wc += list(np.clip(1 - (g[1:] @ g[0:1].T).ravel(), 0, 1))
print('mean within-cluster paraphrase distance: %.3f' % np.mean(wc))